<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_03_feature_engineering/stage_03b_feature_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03b_feature_application**

## **Introducción**

Esta notebook corresponde al stage_03 - Technical Indicators del pipeline neural_profit y tiene como objetivo generar, evaluar y consolidar indicadores técnicos intradía para el índice MNQ, a partir de datos minuto a minuto.

Partiendo del dataset intradía ya etiquetado con objetivos de retorno, se calculan indicadores técnicos de forma independiente por jornada, evitando la mezcla de información entre días. Esto garantiza consistencia temporal y previene leakage en etapas posteriores de modelado.

El proceso incluye la evaluación cuantitativa de los indicadores mediante Information Coefficient (IC), utilizando correlación de Spearman entre cada indicador y los targets de retorno definidos para distintos horizontes. Este análisis permite medir no solo la relación promedio con el target, sino también su estabilidad a lo largo del tiempo.

Como resultado final, se generan datasets consolidados y listos para modelado, que incluyen:

- Variables OHLCV
- Targets de retorno a distintos horizontes
- Indicadores técnicos seleccionados y validados

Estos artefactos serán utilizados en las siguientes etapas del pipeline para selección de features, entrenamiento y evaluación de modelos predictivos.

## **0. Configuración del Entorno**


## 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 0.2. Instalación e importación de librerías


In [3]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos

import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal



from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

In [4]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

  Preparing metadata (setup.py) ... done
Librería instalada: technical-analysis


In [5]:
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

## 0.4. Definición de rutas



In [6]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [7]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/targets/mnq_intraday_targets.parquet"))
#IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/features/mnq_features_target.parquet"))
#OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_04_feature_engineering_summary.json"))

In [8]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
#IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
#OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

## 0.5. Códigos auxiliares para carga de datos y visualización


In [9]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [10]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")

In [11]:
mnq_intraday_targets = load_mnq_parquet()
info = mnq_dataset_info(mnq_intraday_targets, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_intraday
Shape: (744013, 21)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_closed', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2019-12-23 06:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2019-12-23  ->  2025-06-13
Total days (trading): 1303
Time-of-day range (minutes): {'min_minute_of_day': 390, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2019-12-23 11:30:00+00:00  ->  2025-06-13 20:00:00+00:00


# **1. Indicadores Técnicos**

Los indicadores técnicos calculados en cada jornada tienen como objetivo capturar dinámicas intradía relevantes del precio y el volumen, tales como momentum, sobrecompra/sobreventa, presión institucional o posibles reversiones. Cada uno aporta información complementaria sobre el comportamiento del mercado a corto plazo. En particular:

## **1.1. Indicadores técnicos individuales**

In [16]:
import pandas as pd
from typing import List, Tuple, Iterable
from ta.volatility import AverageTrueRange
from ta.momentum import ROCIndicator

def calcular_indicadores_tecnicos(
    df: pd.DataFrame,
    *,
    target: str = "close",
    date_col: str = "date",
    momentum_windows: Tuple[int, int] = (10, 5),
    ema_span: int = 60,
    atr_windows: Iterable[int] = (14, 20),
    roc_windows: Tuple[int, int, int] = (20, 30, 60),
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Calcula indicadores técnicos por día (groupby(date_col)):
      - Momentum: pct_change(w)
      - EMA normalizada: close / EMA(span) - 1
      - ATR normalizado: ATR(w) / close
      - ROC: ROC(w)

    Retorna:
      (df_con_indicadores, indicator_columns)
    """
    # -------------------------
    # Columnas a crear
    # -------------------------
    mom_cols = [f"mom_{w}" for w in momentum_windows]
    ema_cols = [f"ema_{ema_span}"]
    atr_cols = [f"atr_norm_{w}" for w in atr_windows]
    roc_cols = [f"roc_{w}" for w in roc_windows]

    indicator_columns = mom_cols + ema_cols + atr_cols + roc_cols

    # -------------------------
    # Cálculo por día
    # -------------------------
    def aplicar_por_dia(grupo: pd.DataFrame) -> pd.DataFrame:
        grupo = grupo.copy()

        # Momentum
        for w in momentum_windows:
            grupo[f"mom_{w}"] = grupo[target].pct_change(w)

        # EMA normalizada
        grupo[f"ema_{ema_span}"] = grupo[target] / grupo[target].ewm(span=ema_span).mean() - 1

        # ATR normalizado
        for w in atr_windows:
            atr = AverageTrueRange(
                high=grupo["high"],
                low=grupo["low"],
                close=grupo[target],
                window=int(w),
            )
            grupo[f"atr_norm_{w}"] = atr.average_true_range() / grupo[target]

        # ROC
        for w in roc_windows:
            grupo[f"roc_{w}"] = ROCIndicator(close=grupo[target], window=int(w)).roc()

        return grupo

    out = df.groupby(date_col, group_keys=False).apply(aplicar_por_dia)
    return out, indicator_columns


## **1.2. Cálculo de indicadores técnicos**

Calculamos los indicadores técnicos

In [17]:
import pandas as pd

# ============================================================
# Calcular indicadores
# ============================================================

mnq_intraday_with_indicators = mnq_intraday_targets.copy()
mnq_intraday_with_indicators, indicator_columns = calcular_indicadores_tecnicos(
    mnq_intraday_with_indicators,
    target="close",
    date_col="date",
    momentum_windows=(10, 5),
    ema_span=60,
    atr_windows=(14, 20),
    roc_windows=(20, 30, 60),
)

# Asegurar índice datetime (si aplica)
mnq_intraday_with_indicators.index = pd.to_datetime(mnq_intraday_with_indicators.index)

In [20]:
info_mnq_indicators = mnq_dataset_info(mnq_intraday_with_indicators, name="mnq_intraday_with_indicators", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_indicators)

Dataset: mnq_intraday_with_indicators
Shape: (744013, 29)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_closed', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90', 'mom_10', 'mom_5', 'ema_60', 'atr_norm_14', 'atr_norm_20', 'roc_20', 'roc_30', 'roc_60']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2019-12-23 06:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2019-12-23  ->  2025-06-13
Total days (trading): 1303
Time-of-day range (minutes): {'min_minute_of_day': 390, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2019-12-23 11:30:00+00:00  ->  2025-06-13 20:00:00+00:00


Filtramos todos los NaNs del dataset

In [21]:
# Eliminación de filas con NaN
mnq_intraday_with_indicators = mnq_intraday_with_indicators.dropna()

In [22]:
info_mnq_indicators = mnq_dataset_info(mnq_intraday_with_indicators, name="mnq_intraday_with_indicators", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_indicators)

Dataset: mnq_intraday_with_indicators
Shape: (548563, 29)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_closed', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90', 'mom_10', 'mom_5', 'ema_60', 'atr_norm_14', 'atr_norm_20', 'roc_20', 'roc_30', 'roc_60']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2019-12-23 07:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2019-12-23  ->  2025-06-13
Total days (trading): 1303
Time-of-day range (minutes): {'min_minute_of_day': 450, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2019-12-23 12:30:00+00:00  ->  2025-06-13 18:30:00+00:00


In [23]:
assert not mnq_intraday_with_indicators.isna().any().any(), \
    "El dataset contiene NaN"

In [24]:
# Verificación posterior
start_time_full_day = info_mnq_indicators["datetime_min"][11:16]
final_time_full_day = info_mnq_indicators["datetime_max"][11:16]

# **2. Introducción de interacciones**

## **2.1. Implementación**

In [25]:
def add_interactions(
    df: pd.DataFrame,
    *,
    date_col: str = "date",
    # Indicadores base (se asume que ya existen en df)
    ema_col: str = "ema_60",
    roc60_col: str = "roc_60",
    roc20_col: str = "roc_20",
    roc30_col: str = "roc_30",
    mom5_col: str = "mom_5",
    mom10_col: str = "mom_10",
    atr14_col: str = "atr_norm_14",
    atr20_col: str = "atr_norm_20",
    # Nombres de salida
    prefix: str = "",
    # Opcional: controlar si crear atr-interactions
    add_atr_interactions: bool = True,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Crea interacciones mínimas, interpretables y compatibles con tu set final:
      - Full day: ema_60, roc_60 (+ atr_norm_20 en H=90)
      - Gestation: ema_60, roc_60, roc_20, roc_30, mom_10, mom_5
      - Execution: ema_60, roc_60, roc_20, roc_30, atr_norm_14

    Importante:
      - Calcula interacciones para TODO el dataset (toda la jornada).
      - Luego se evalúan IC/ventana filtrando por horario en tu pipeline.
      - Lags y slope se calculan por día (groupby(date).shift(1)) => sin leakage.

    Devuelve:
      - df_out: dataframe con nuevas columnas
      - created_cols: lista con los nombres creados
    """
    df_out = df.copy()

    # -------------------------
    # Helper para nombres
    # -------------------------
    def _col(name: str) -> str:
        return f"{prefix}{name}" if prefix else name

    # -------------------------
    # Validaciones mínimas (solo lo imprescindible)
    # -------------------------
    needed = [date_col, ema_col, roc60_col]
    missing = [c for c in needed if c not in df_out.columns]
    if missing:
        raise KeyError(f"Faltan columnas requeridas para interacciones: {missing}")

    created_cols: list[str] = []

    # ATR x ROC (contexto de volatilidad)
    # - atr_norm_20: más coherente para full_day H=90
    # - atr_norm_14: más coherente para execution
    if add_atr_interactions:
        if atr20_col in df_out.columns:
            roc60_x_atr20 = _col("roc60_x_atr20")
            df_out[roc60_x_atr20] = df_out[roc60_col] * df_out[atr20_col]
            created_cols.append(roc60_x_atr20)

        if atr14_col in df_out.columns:
            roc60_x_atr14 = _col("roc60_x_atr14")
            df_out[roc60_x_atr14] = df_out[roc60_col] * df_out[atr14_col]
            created_cols.append(roc60_x_atr14)

    # -------------------------
    # 2) Gestation (08:00-09:00): roc_20/roc_30 y mom_5/mom_10
    #    (se crean para todo el día; se evalúan en ventana)
    # -------------------------
    if roc20_col in df_out.columns:
        roc20_minus_roc60 = _col("roc20_minus_roc60")
        df_out[roc20_minus_roc60] = df_out[roc20_col] - df_out[roc60_col]
        created_cols.append(roc20_minus_roc60)


    if (mom5_col in df_out.columns) and (mom10_col in df_out.columns):
        mom5_minus_mom10 = _col("mom5_minus_mom10")
        df_out[mom5_minus_mom10] = df_out[mom5_col] - df_out[mom10_col]
        created_cols.append(mom5_minus_mom10)

    return df_out, created_cols


## **2.1. Aplicación**

In [26]:
mnq_with_interactions, interaction_cols = add_interactions(
    mnq_intraday_with_indicators,
    date_col="date",
    ema_col="ema_60",
    roc60_col="roc_60",
    roc20_col="roc_20",
    roc30_col="roc_30",
    mom5_col="mom_5",
    mom10_col="mom_10",
    atr14_col="atr_norm_14",
    atr20_col="atr_norm_20",
)

print("Interacciones creadas:")
print(interaction_cols)


Interacciones creadas:
['roc60_x_atr20', 'roc60_x_atr14', 'roc20_minus_roc60', 'mom5_minus_mom10']


In [29]:
mnq_with_interactions.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day',
       'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_closed',
       'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60',
       'delta_90', 'ret_90', 'mom_10', 'mom_5', 'ema_60', 'atr_norm_14',
       'atr_norm_20', 'roc_20', 'roc_30', 'roc_60', 'roc60_x_atr20',
       'roc60_x_atr14', 'roc20_minus_roc60', 'mom5_minus_mom10'],
      dtype='object')

# **3. Features base**

In [30]:
mnq_with_interactions.drop(columns=["open", "high", "low", "volume"], inplace=True)

In [31]:
mnq_with_interactions.columns

Index(['date', 'close', 'minute_of_day', 'is_premarket', 'is_opening',
       'is_regular', 'is_closing', 'is_closed', 'is_mon', 'is_tue', 'is_wed',
       'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90',
       'mom_10', 'mom_5', 'ema_60', 'atr_norm_14', 'atr_norm_20', 'roc_20',
       'roc_30', 'roc_60', 'roc60_x_atr20', 'roc60_x_atr14',
       'roc20_minus_roc60', 'mom5_minus_mom10'],
      dtype='object')

# **9. Consolidación de features**

**Target: `delta_60`**

| delta_60  | atr_norm_14 | atr_norm_20 | ema_60 | mom_10 | mom_5 | roc_20 | roc_30 | roc_60 | roc60_x_atr20 | roc60_x_atr14 | roc20_minus_roc60 | mom5_minus_mom10 |
|------------|-------------|-------------|--------|--------|--------|--------|--------|--------|----------------|----------------|-------------------|-------------------|
| full day   | 1 | 1 | 1 | 0 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |
| gestation  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 0 | 1 | 1 | 1 | 1 |
| execution  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |

**Target: `delta_90`**

| delta_90  | atr_norm_14 | atr_norm_20 | ema_60 | mom_10 | mom_5 | roc_20 | roc_30 | roc_60 | roc60_x_atr20 | roc60_x_atr14 | roc20_minus_roc60 | mom5_minus_mom10 |
|------------|-------------|-------------|--------|--------|--------|--------|--------|--------|----------------|----------------|-------------------|-------------------|
| full day   | 1 | 1 | 1 | 0 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |
| gestation  | 1 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 0 |
| execution  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |

**Target: `ret_60`**

| ret_60    | atr_norm_14 | atr_norm_20 | ema_60 | mom_10 | mom_5 | roc_20 | roc_30 | roc_60 | roc60_x_atr20 | roc60_x_atr14 | roc20_minus_roc60 | mom5_minus_mom10 |
|------------|-------------|-------------|--------|--------|--------|--------|--------|--------|----------------|----------------|-------------------|-------------------|
| full day   | 1 | 1 | 1 | 0 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |
| gestation  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 0 | 0 | 0 | 1 | 1 |
| execution  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |


**Target: `ret_90`**

| ret_90    | atr_norm_14 | atr_norm_20 | ema_60 | mom_10 | mom_5 | roc_20 | roc_30 | roc_60 | roc60_x_atr20 | roc60_x_atr14 | roc20_minus_roc60 | mom5_minus_mom10 |
|------------|-------------|-------------|--------|--------|--------|--------|--------|--------|----------------|----------------|-------------------|-------------------|
| full day   | 1 | 1 | 1 | 0 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |
| gestation  | 1 | 1 | 0 | 0 | 0 | 0 | 0 | 1 | 1 | 1 | 1 | 0 |
| execution  | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |



## **9.1. Justificación del set final de features**


A partir del análisis estadístico, estructural y de estabilidad realizado a lo largo de este stage, se define un conjunto de features que equilibra **capacidad predictiva, estabilidad out-of-sample e interpretabilidad**, evitando redundancia y sobreajuste.

**1. Variable OHLC seleccionada**

  - **`close`**

    - Las variables OHLC presentan **correlación extremadamente alta entre sí** (> 0.99), por lo que aportan información prácticamente redundante.
    - `close` concentra la información relevante del precio:
      - Es el valor más utilizado por los indicadores técnicos.
      - Resume el consenso del mercado al cierre de cada minuto.
    - Su IC es **estable entre horizontes (H=60 y H=90)** y consistente entre IS y OOS.

  - **Decisión**: se utiliza únicamente `close` como variable de precio base.

**2. Indicadores técnicos seleccionados**

Los indicadores fueron elegidos por cumplir simultáneamente:

  - IC out-of-sample distinto de cero,
  - coherencia direccional IS vs OOS,
  - estabilidad entre horizontes o especialización explicable,
  - alineación con la lógica de mercado (tendencia + momentum).

  Los indicadores técnicos seleccionados son:

  - `ema_60`

    - Captura la **tendencia intradía dominante**.
    - Es estable en full day y relevante en ventanas operativas.
    - Funciona como **filtro estructural**, no como señal puntual.

  - `roc_20`, `roc_30`, `roc_60`

    - Representan **velocidad y magnitud del movimiento del precio** en distintas escalas.
    - `roc_60`: estructura de medio plazo (usable todo el día).
    - `roc_20` y `roc_30`: dinámicas más cortas, relevantes en ventanas específicas.
    - Muestran **buena generalización OOS** y coherencia entre horizontes.

  - `momentum_3`, `momentum_5`, `momentum_10`

    - Capturan impulsos de **muy corto plazo**, especialmente útiles en ventanas de gestación y ejecución.
    - Individualmente pueden ser ruidosos, pero:
      - mantienen señal consistente en contextos específicos,
      - son la base para interacciones más informativas.

**3. Interacciones seleccionadas**

Las interacciones no se introducen para aumentar complejidad, sino para **capturar relaciones relativas entre escalas temporales**, que mostraron mayor estabilidad que algunos indicadores crudos.

  - `mom3_minus_mom10`

    - Mide aceleración corta vs impulso base.
    - Detecta cambios tempranos de régimen.
    - Mostró buena estabilidad en execution.

  - `mom5_minus_mom10`

    - Captura divergencias entre momentum corto y medio.
    - Útil para distinguir continuidad vs agotamiento.

  - `mom3_minus_mom5`

    - Refina el análisis del impulso inmediato.
    - Aporta señal complementaria en ventanas operativas.

Las interacciones **no reemplazan** a los indicadores base, sino que los **contextualizan**.

**4. Cómo se utilizarán en el modelo**

- **Indicadores base (`ema`, `roc`, `momentum`)**:
  - Proveen información primaria de tendencia y velocidad.
  - Se utilizarán como features directos (eventualmente normalizados intradía).

- **Interacciones de momentum**:
  - Actúan como features de **contexto relativo**.
  - Ayudan al modelo a distinguir:
    - impulso genuino vs ruido,
    - aceleración vs desaceleración.

- **Separación por ventanas (full day / gestation / execution)**:
  - Los features se calculan para todo el día,
  - pero su **evaluación y peso** se interpretan según la ventana donde demostraron mayor valor.

## **9.2. Decisión final y set consolidado de features**


El set final de features:

- es **compacto** (sin redundancias),
- **estable out-of-sample**,
- **interpretable desde la lógica de mercado**,
- y está preparado para **modelos que generalicen**, no para optimizar IC in-sample.

Este conjunto constituye una base sólida para el entrenamiento de modelos intradía con control explícito de generalización y sin leakage.

| Tipo de feature                | Variable(s)                                      | Rol económico principal                                                                 |
|--------------------------------|--------------------------------------------------|------------------------------------------------------------------------------------------|
| Precio intradía                | `close`                                         | Estado representativo del precio; consenso del mercado en cada minuto                    |
| Tendencia intradía             | `ema_60`                                        | Dirección y nivel tendencial dominante del día; filtro estructural del mercado           |
| Momentum de muy corto plazo    | `momentum_3`, `momentum_5`                      | Impulso inmediato; captura aceleraciones y micro-movimientos del precio                 |
| Momentum de corto plazo        | `momentum_10`                                   | Intensidad del desplazamiento reciente; referencia base de impulso                       |
| Velocidad de corto plazo       | `roc_20`, `roc_30`                              | Rapidez y magnitud del movimiento en escalas cortas; útil en ventanas operativas         |
| Velocidad de medio plazo       | `roc_60`                                       | Dinámica estructural del movimiento; estabilidad direccional a lo largo del día          |
| Interacción de momentum        | `mom3_minus_mom10`                              | Aceleración relativa: impulso corto frente a impulso base                                |
| Interacción de momentum        | `mom5_minus_mom10`                              | Divergencia entre momentum corto y medio; detección de agotamiento o continuidad         |
| Interacción de momentum        | `mom3_minus_mom5`                               | Refinamiento del impulso inmediato; cambios tempranos en la dinámica intradía            |


## **9.3. Implicancia para el pipeline**


Este set consolidado constituye el núcleo de features técnicas del modelo y será utilizado en las etapas posteriores para:

- entrenamiento y validación del modelo predictivo,
- análisis de contribución por feature,
- y evaluación de desempeño económico.

La consolidación reduce la dimensionalidad, mejora la interpretabilidad y alinea el diseño del modelo con la estructura temporal y económica observada en los datos.

## **9.4. Implementación**


In [ ]:
mnq_intraday_labeled = load_data()

# Asumimos que el índice es DatetimeIndex tz-aware
idx = mnq_intraday_labeled.index

mnq_intraday_labeled["minute_of_day"] = idx.hour * 60 + idx.minute

NameError: name 'load_data' is not defined

In [ ]:
def info_dataset_final(df):
    print("Información del dataset:\n")

    # -------------------------
    # Días y registros
    # -------------------------
    num_dias = df["date"].nunique()
    print(f"\tCantidad de días: {num_dias}")

    validos_por_dia = (
        df.dropna(subset=["close"])
          .groupby("date")
          .size()
    )
    promedio_por_fecha = validos_por_dia.mean()
    print(f"\tRegistros por día: {int(promedio_por_fecha)}")

    # -------------------------
    # Horarios (desde minute_of_day)
    # -------------------------
    if "minute_of_day" in df.columns:
        min_minute = int(df["minute_of_day"].min())
        max_minute = int(df["minute_of_day"].max())

        primer_hora = f"{min_minute // 60:02d}:{min_minute % 60:02d}"
        ultima_hora = f"{max_minute // 60:02d}:{max_minute % 60:02d}"
    else:
        primer_hora = None
        ultima_hora = None

    print(f"\tHora diaria de inicio: {primer_hora}")
    print(f"\tHora diaria de final: {ultima_hora}")

    # -------------------------
    # Columnas del dataset
    # -------------------------
    print("\n\tColumnas del dataset:")
    for c in df.columns:
        print(f"\t- {c}")



In [ ]:
info_dataset_final(mnq_intraday_labeled)

In [ ]:
import pandas as pd
from typing import List, Tuple
from ta.momentum import ROCIndicator


def build_mnq_features_targets_full(
    mnq_intraday_labeled: pd.DataFrame,
    *,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    close_col: str = "close",
    target_cols: Tuple[str, str] = ("ret_60", "ret_90"),
    tz: str | None = "America/New_York",   # se deja, pero NO se usa para el índice
    keep_datetime_col: bool = True,
) -> Tuple[pd.DataFrame, List[str], List[str]]:

    # ------------------------------------------------------------
    # Preservar el índice original (NO tocarlo)
    # ------------------------------------------------------------
    original_index = mnq_intraday_labeled.index

    # ------------------------------------------------------------
    # 0) Copiar dataset
    # ------------------------------------------------------------
    df = mnq_intraday_labeled.copy()

    # -------------------------
    # 0) Validación columnas mínimas
    # -------------------------
    required_base = [date_col, minute_col, close_col, *list(target_cols)]
    missing = [c for c in required_base if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas en mnq_intraday_labeled: {missing}")

    # ------------------------------------------------------------
    # 1) Normalización de nombres (si vinieran con delta_pts_*)
    # ------------------------------------------------------------
    #df = df.rename(columns={"ret_60": "ret60", "ret_90": "ret90"})

    # ------------------------------------------------------------
    # 2) Indicadores técnicos (por día)
    # ------------------------------------------------------------
    technical_indicators_features: List[str] = [
        "ema60",
        "mom3",
        "mom5",
        "mom10",
        "roc20",
        "roc30",
        "roc60",
    ]

    def _apply_per_day(g: pd.DataFrame) -> pd.DataFrame:
        g = g.copy()

        g["ema60"] = g[close_col] / g[close_col].ewm(span=60, adjust=False).mean() - 1.0

        g["mom3"] = g[close_col].pct_change(3)
        g["mom5"] = g[close_col].pct_change(5)
        g["mom10"] = g[close_col].pct_change(10)

        g["roc20"] = ROCIndicator(close=g[close_col], window=20).roc()
        g["roc30"] = ROCIndicator(close=g[close_col], window=30).roc()
        g["roc60"] = ROCIndicator(close=g[close_col], window=60).roc()

        return g

    # Groupby por date (columna), no por índice
    df = df.groupby(df[date_col], group_keys=False, sort=False).apply(_apply_per_day)

    # ✅ Garantía dura: restaurar el índice EXACTO del input
    # (si por cualquier motivo apply lo alteró)
    if not df.index.equals(original_index):
        df = df.copy()
        df.index = original_index

    # ------------------------------------------------------------
    # 3) Interacciones mínimas
    # ------------------------------------------------------------
    def calculate_interactions_features(
        df_in: pd.DataFrame,
        *,
        date_col: str = "date",
        minute_col: str = "minute_of_day",
        mom3_col: str = "mom3",
        mom5_col: str = "mom5",
        mom10_col: str = "mom10",
        prefix: str = "",
    ) -> Tuple[pd.DataFrame, List[str]]:

        df_out = df_in.copy()

        def _col(name: str) -> str:
            return f"{prefix}{name}" if prefix else name

        needed = [date_col, minute_col, mom3_col, mom5_col, mom10_col]
        miss = [c for c in needed if c not in df_out.columns]
        if miss:
            raise KeyError(f"Faltan columnas requeridas para interacciones: {miss}")

        created_cols: List[str] = []

        c = _col("mom3_mom10")
        df_out[c] = df_out[mom3_col] - df_out[mom10_col]
        created_cols.append(c)

        c = _col("mom5_mom10")
        df_out[c] = df_out[mom5_col] - df_out[mom10_col]
        created_cols.append(c)

        c = _col("mom3_mom5")
        df_out[c] = df_out[mom3_col] - df_out[mom5_col]
        created_cols.append(c)

        return df_out, created_cols

    df, interactions_features = calculate_interactions_features(
        df,
        date_col=date_col,
        minute_col=minute_col,
        mom3_col="mom3",
        mom5_col="mom5",
        mom10_col="mom10",
    )


    # ------------------------------------------------------------
    # 4) Dataset final (selección de columnas)
    # ------------------------------------------------------------
    final_cols: List[str] = [
        date_col,
        minute_col,
        close_col,
        *technical_indicators_features,
        *interactions_features,
        *list(target_cols),
    ]

    # Mantengo esto solo si la columna "datetime" EXISTE; si no, no la invento
    if keep_datetime_col and "datetime" in df.columns:
        final_cols = ["datetime"] + final_cols

    missing_final = [c for c in final_cols if c not in df.columns]
    if missing_final:
        raise ValueError(f"No se pudieron construir todas las columnas finales: {missing_final}")

    mnq_features_targets = df.loc[:, final_cols].copy()

    # Garantía final: el índice sale idéntico al que entró
    if not mnq_features_targets.index.equals(original_index):
        mnq_features_targets.index = original_index

    # ------------------------------------------------------------
    # 5) Renombrar targets al final: ret_* -> ret*
    # ------------------------------------------------------------
    rename_targets = {"ret_60": "ret60", "ret_90": "ret90"}
    mnq_features_targets = mnq_features_targets.rename(columns=rename_targets)

    targets = ['ret60', 'ret90']
    # Si desea que target_cols también salga coherente (opcional)
    # target_cols_out = tuple(rename_targets.get(c, c) for c in target_cols)

    return mnq_features_targets, technical_indicators_features, interactions_features, targets


In [ ]:
mnq_features_targets, tech_feats, inter_feats, targets = build_mnq_features_targets_full(
    mnq_intraday_labeled=mnq_intraday_labeled
)

In [ ]:
#mnq_features_targets.head(15)

In [ ]:
print(f"OHLVC feats: ['close'] \n")
print(f"Tech feats: {tech_feats} \n")
print(f"Interaction feats: {inter_feats} \n")
print(f"Targets: {targets}")

In [ ]:
info_dataset_final(mnq_features_targets)

# **10. Flags de activación por ventanas operativas**

## **10.1. Introducción conceptual**


**1. Motivación principal: valor predictivo dependiente del horario**

El dataset incluye features cuyo valor predictivo no es homogéneo a lo largo de la jornada.

En particular:

- Algunos indicadores (por ejemplo, ciertos momentum y ROC) muestran mayor capacidad predictiva durante las ventanas de:

  - gestation (08:00-09:00)
  - execution (09:00-10:00)

- Fuera de esas ventanas, esas mismas features:
  - pierden señal,
  - se vuelven ruidosas,
  - o directamente dejan de ser informativas para la toma de decisiones.

Sin embargo, el modelo se entrena usando ventanas deslizantes a lo largo de todo el full day.
Por lo tanto, es necesario informarle explícitamente al modelo en qué contextos temporales una feature es relevante.

**2. Por qué se utilizan flags y no filtros duros**

En lugar de:

- eliminar features fuera de ciertas horas, o
- entrenar modelos distintos por franja horaria,

se introducen flags de activación para permitir que el modelo:

- vea toda la jornada (maximizando datos),
- pero aprenda cuándo una feature es confiable.

Cada flag responde a la pregunta:

>“¿Esta feature tiene sentido predictivo en este momento del día?”

**3. Qué aprende el modelo con este esquema**

Cada muestra contiene pares del tipo:

$$  
𝑋_t = [roc20, roc20_{active}, mom3, mom3_{active}, ... ]
$$

Durante el entrenamiento, el modelo aprende patrones como:

- Cuando `roc20_active` = 1

  → `roc20` suele correlacionar mejor con el target.

- Cuando `roc20_active` = 0

  → `roc20` pierde poder explicativo y debe ser atenuado o ignorado.

De este modo, el modelo modula dinámicamente la importancia de cada feature en función del contexto horario, sin reglas hard-coded.

**4. Por qué no es suficiente “poner ceros”**

Asignar 0 a una feature fuera de su ventana introduce ambigüedad:

- `roc20` = 0 puede significar:

  - un valor legítimo del indicador, o
  - una feature fuera de su régimen predictivo.

Con flags explícitos:
  - `roc20` = 0
  - `roc20_active` = 0

el modelo puede distinguir claramente entre:
  - “valor bajo pero válido”
  - “valor no relevante en este horario”

**5. Interpretación conceptual**

El flag actúa como un contextualizador temporal, no como una regla.

- Feature → señal numérica
- Flag → indica si la señal está en su régimen de mayor valor predictivo

El modelo aprende implícitamente:

> “Esta feature solo merece atención cuando estoy dentro de la ventana operativa adecuada.”

**6. Condiciones necesarias para que el enfoque sea válido**

1. Cada feature dependiente del horario tiene su flag correspondiente.
2. Los flags se mantienen como variables binarias (0/1) y no se escalan.
3. El orden de columnas es estrictamente consistente en todo el pipeline.

**Conclusión**

El uso de flags permite entrenar un modelo con cobertura completa del día, sin perder la capacidad de:

- capturar regímenes horarios de mayor valor predictivo,
- diferenciar señal estructural de ruido intradía,
- y alinear el aprendizaje con las ventanas reales de operación y ejecución.

Este diseño es coherente con un enfoque full-day learning y window-aware decision making.

## **10.2. Implementación**


In [ ]:
execution_features = ['mom3', 'mom5', 'roc20', 'mom3_mom5']
gestation_features = ['mom10','roc30','mom3_mom10','mom5_mom10'] + execution_features

In [ ]:
#print(f'gestation_window: {start_time_gestation_window}-{final_time_gestation_window}')
#print(f'gestation_features: {gestation_features}')

#print(f'execution_window: {start_time_execution_window}-{final_time_execution_window}')
#print(f'execution_features: {execution_features}' )

In [ ]:
windows = {
    "gestation": (start_time_gestation_window, final_time_gestation_window),
    "execution": (start_time_execution_window, final_time_execution_window)
}

window_features = {
    "gestation": gestation_features,
    "execution": execution_features,
}

In [ ]:
import pandas as pd
from typing import Iterable, Dict, Tuple

def add_time_window_feature_flags(
    df: pd.DataFrame,
    *,
    windows: Dict[str, Tuple[str, str]],
    window_features: Dict[str, Iterable[str]],
    flag_suffix: str = "_active",
) -> pd.DataFrame:
    """
    Agrega flags binarios por feature indicando si el timestamp (índice datetime)
    cae dentro de la ventana temporal asociada a esa feature.

    - NO modifica el índice.
    - NO altera valores de las features.
    - Devuelve una copia del DataFrame con nuevas columnas *_active.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset con índice DatetimeIndex.
    windows : dict
        Mapeo nombre_ventana -> (start_time, end_time) en formato 'HH:MM'.
        Ej: {'gestation': ('08:00','09:00'), 'execution': ('09:00','10:00')}
    window_features : dict
        Mapeo nombre_ventana -> iterable de features que aplican a esa ventana.
    flag_suffix : str
        Sufijo para las columnas flag. Default '_active'.

    Retorna
    -------
    pd.DataFrame
        Copia del df con flags agregados.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El DataFrame debe tener un índice DatetimeIndex.")

    out = df.copy()

    # Hora del índice sin tocar timezone
    idx_time = out.index.strftime("%H:%M")

    for window_name, (start, end) in windows.items():
        if window_name not in window_features:
            continue

        is_in_window = (idx_time >= start) & (idx_time < end)

        for feat in window_features[window_name]:
            if feat not in out.columns:
                raise ValueError(f"La feature '{feat}' no existe en el DataFrame.")

            flag_col = f"{feat}{flag_suffix}"
            # Si la feature aparece en múltiples ventanas, OR lógico
            if flag_col in out.columns:
                out[flag_col] = out[flag_col] | is_in_window.astype("int8")
            else:
                out[flag_col] = is_in_window.astype("int8")

    return out



In [ ]:
mnq_features_targets = add_time_window_feature_flags(
    mnq_features_targets,
    windows=windows,
    window_features=window_features,
)

In [ ]:
info_dataset_final(mnq_features_targets)

In [ ]:
mnq_features_targets.columns

In [ ]:
base_order = [
    'date',
    'minute_of_day',
]

features_order = [
    'close',
    'ema60',
    'roc60',

    'roc30', 'roc30_active',
    'roc20', 'roc20_active',

    'mom10', 'mom10_active',
    'mom5',  'mom5_active',
    'mom3',  'mom3_active',

    'mom5_mom10', 'mom5_mom10_active',
    'mom3_mom10', 'mom3_mom10_active',
    'mom3_mom5',  'mom3_mom5_active',
]

target_order = [
    'ret60',
    'ret90',
]

In [ ]:
import pandas as pd
from typing import List

def reorder_mnq_columns(
    df: pd.DataFrame,
    *,
    base_order: List[str],
    features_order: List[str],
    target_order: List[str],
) -> pd.DataFrame:
    """
    Reordena las columnas del dataset mnq_features_targets según un orden explícito.

    - NO modifica el índice.
    - NO altera valores.
    - Valida que todas las columnas requeridas existan.
    - Devuelve una copia del DataFrame con el nuevo orden.

    Orden final:
        base_order + features_order + target_order
    """
    required_cols = base_order + features_order + target_order

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Faltan columnas requeridas en el DataFrame: {missing}"
        )

    return df[required_cols].copy()


In [ ]:
mnq_features_targets = reorder_mnq_columns(
    mnq_features_targets,
    base_order=[
        'date',
        'minute_of_day',
    ],
    features_order=[
        'close',
        'ema60',
        'roc60',

        'roc30', 'roc30_active',
        'roc20', 'roc20_active',

        'mom10', 'mom10_active',
        'mom5',  'mom5_active',
        'mom3',  'mom3_active',

        'mom5_mom10', 'mom5_mom10_active',
        'mom3_mom10', 'mom3_mom10_active',
        'mom3_mom5',  'mom3_mom5_active',
    ],
    target_order=[
        'ret60',
        'ret90',
    ],
)


In [ ]:
mnq_features_targets

In [ ]:
OUT_PARQUET

In [ ]:
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

# Guardar dataset
mnq_features_targets.to_parquet(OUT_PARQUET, index=True)